In [1]:
import pandas as pd
import pyarrow

In [2]:
url = "C:/Users/Michael/Downloads/popec22.parquet"

df = pd.read_parquet(url, engine='pyarrow')

excluir = ['P00','P01','P02','P03']

agrupacion = [c for c in df.columns if c not in excluir]

# Al obtener el número máximo del número de persona obtendremos la cantidad de personas del hogar
personasporhogar = (
    df.groupby(agrupacion)
      .agg(numpersonas=('P00','max'))
      .reset_index()
)

personasporhogar = personasporhogar[personasporhogar['INH']>0]
personasporhogar

,I01,I02,I03,I04,I05,I10,INH,numpersonas
0,1,1,50,1,1,1,1,5
1,1,1,50,1,1,2,1,4
2,1,1,50,1,1,3,1,4
3,1,1,50,1,1,4,1,4
4,1,1,50,1,1,5,1,4
...,...,...,...,...,...,...,...,...
2784857,24,3,52,999,4,70,1,3
2784858,24,3,52,999,4,71,1,2
2784859,24,3,52,999,4,72,1,1
2784860,24,3,52,999,4,73,1,1


In [3]:
parroquia = (
    personasporhogar[
        (df['I01'] == 1 ) & 
        (df['I02'] == 1) &  
        (df['I03'] == 67)
    ]
)

C:\Users\Michael\AppData\Local\Temp\ipykernel_14192\3232994116.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  personasporhogar[


In [4]:
# Provincia: Sucumbíos, Cantón: Sucumbíos, Parroquia: La Sofía
# código: 210552
parroquia = personasporhogar[
    (personasporhogar['I01'] == 21) & 
    (personasporhogar['I02'] == 5) & 
    (personasporhogar['I03'] == 52)
]

# Obtenemos la probabilidad por tamaño de hogar
freq_parroquia = (
    parroquia['numpersonas']
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)

freq_parroquia

,numpersonas,proportion
0,1,0.210526
1,2,0.157895
2,3,0.263158
3,4,0.105263
4,6,0.157895
5,7,0.052632
6,9,0.052632


In [5]:
import altair as alt

chart = alt.Chart(freq_parroquia).mark_bar(
    size=25
).encode(
    x=alt.X(
        'numpersonas:O',
        sort=None,
        title='Número de personas en el hogar',
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        'proportion:Q',
        title='Porcentaje de hogares',
        axis=alt.Axis(format='.0%')
    ),
    tooltip=[
        alt.Tooltip('numpersonas:O', title='Número de miembros'), 	
        alt.Tooltip('proportion:Q', title='Probabilidad del evento', format='.2%')
    ]
).properties(
    title="Distribución del tamaño del hogar",
    width=700,
    height=400
)

chart

alt.Chart(...)

In [6]:
# Probabilidades de la población
freq_df = (
    personasporhogar['numpersonas']
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)


# Unir distribuciones
freq = freq_df.rename(columns={'proportion': 'poblacion'}).merge(
    freq_parroquia.rename(columns={'proportion': 'muestra'}), 
    on='numpersonas', how='left'
)

freq

,numpersonas,poblacion,muestra
0,1,1.130299e-01,0.210526
1,2,1.552242e-01,0.157895
2,3,1.931299e-01,0.263158
3,4,2.279615e-01,0.105263
4,5,1.625016e-01,NaN
5,6,8.058090e-02,0.157895
6,7,3.631705e-02,0.052632
7,8,1.628458e-02,NaN
8,9,7.622012e-03,0.052632
9,10,3.738890e-03,NaN


In [7]:
# Cambiamos la estructura de la tabla
freq_resize = freq.melt(
    id_vars='numpersonas',
    var_name='grupo',
    value_name='porcentaje'
)

In [8]:
chart = alt.Chart(freq_resize).mark_bar(
    size=10 # más ancho (porque hay xOffset)
).encode(
    x=alt.X(
        'numpersonas:O',
        title='Número de personas en el hogar',
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        'porcentaje:Q',
        title='Porcentaje de hogares',
        axis=alt.Axis(format='.1%')
    ),
    color=alt.Color(
        'grupo:N',
        sort=['poblacion','muestra'],
        title='',
        legend=alt.Legend(orient='top'),   # mueve la leyenda arriba
        scale=alt.Scale(
            domain=['poblacion','muestra'],
            range=['#013440', '#593954']
        )
    ),
    xOffset=alt.XOffset(
        'grupo:N',
        sort=['poblacion','muestra']   # importante también aquí
    ),
    tooltip=[
        alt.Tooltip('numpersonas:O', title='Personas'),
        alt.Tooltip('grupo:N', title='Grupo'),
        alt.Tooltip('porcentaje:Q', title='Porcentaje',format='.2%')
    ]
).properties(
    title="Distribución del tamaño del hogar",
    width=700,
    height=400
)

chart

alt.Chart(...)

In [9]:
muestra = personasporhogar.sample(frac=0.1, random_state=123)

In [10]:
# Función que toma una selección aleatoria según tamaño
sizepop = len(personasporhogar)
def comparacion_muestrales(tamano_muestra=100):

    if tamano_muestra > sizepop:
        print(f"El tamaño de la muestra ({tamano_muestra}) es mayor que la población ({sizepop})")
        return

    muestra = personasporhogar.sample(
        n=min(tamano_muestra, sizepop),   # ← protección extra
        random_state=123
    )

    # Obtenemos la probabilidad por tamaño de hogar
    freq_muestra = (
        muestra['numpersonas']
        .value_counts(normalize=True)
        .sort_index()
        .reset_index()
    )

    # Unir distribuciones muestra - población
    freq = freq_df.rename(columns={'proportion': 'poblacion'}).merge(
        freq_muestra.rename(columns={'proportion': 'muestra'}), 
        on='numpersonas', how='left'
    )

    # Cambiamos la estructura de la tabla
    freq_resize = freq.melt(
        id_vars='numpersonas',
        var_name='grupo',
        value_name='porcentaje'
    )

    grafico = alt.Chart(freq_resize).mark_bar(
        size=10 # más ancho (porque hay xOffset)
    ).encode(
        x=alt.X(
            'numpersonas:O',
            title='Número de personas en el hogar',
            axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y(
            'porcentaje:Q',
            title='Porcentaje de hogares',
            axis=alt.Axis(format='.1%')
        ),
        color=alt.Color(
            'grupo:N',
            sort=['poblacion','muestra'],
            title='',
            legend=alt.Legend(orient='top'),   # mueve la leyenda arriba
            scale=alt.Scale(
                domain=['poblacion','muestra'],
                range=['#013440', '#593954']
            )
        ),
        xOffset=alt.XOffset(
            'grupo:N',
            sort=['poblacion','muestra']   # importante también aquí
        ),
        tooltip=[
            alt.Tooltip('numpersonas:O', title='Personas'),
            alt.Tooltip('grupo:N', title='Grupo'),
            alt.Tooltip('porcentaje:Q', title='Porcentaje',format='.2%')
        ]
    ).properties(
        title="Distribución del tamaño del hogar",
        width=700,
        height=400
    )

    return grafico

In [11]:
comparacion_muestrales(tamano_muestra=100)

alt.Chart(...)

In [12]:
import ipywidgets as widgets
from ipywidgets import interact

step_size = 1000

interact(
    comparacion_muestrales,
    tamano_muestra=widgets.IntSlider(
        min=100,
        max=sizepop,
        step=100,
        value=500,
        description='n',
        continuous_update=False
    )
)

interactive(children=(IntSlider(value=500, continuous_update=False, description='n', max=2780237, min=100, ste…

<function __main__.comparacion_muestrales(tamano_muestra=100)>